# Caderno de aprendizagem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/00_aprendizagem.ipynb)

Este caderno vem **antes** de cada notebook do TCC. Os notebooks do TCC fazem a pesquisa em si, para as 370
estações. Aqui você aprende, com exemplos pequenos, cada ideia e cada ferramenta que eles usam, para chegar lá
sabendo o que está acontecendo.

### Como o caderno está organizado

Um capítulo para cada notebook do TCC, na mesma ordem. Cada capítulo tem dois tipos de item:

- **Resolvido**: o exemplo vem pronto, com a explicação antes e os comentários no código. Rode, leia e **mude os
  valores** para ver o que acontece;
- **Para pensar e resolver**: perguntas para responder com código ou com texto, na célula vazia logo abaixo.

As respostas dos itens "para pensar e resolver" são a pauta das reuniões com o orientador.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula só define DE ONDE os dados serão lidos.
# No Colab, os dados são lidos direto do GitHub (não precisa baixar nada).
# No computador do orientador, dentro da pasta do projeto, ela usa a pasta dados/ local.
from pathlib import Path

if Path("../dados").exists():
    DADOS = "../dados/"                     # rodando dentro da pasta do projeto
else:
    DADOS = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/dados/"

print("Os dados serão lidos de:", DADOS)

In [ ]:
import pandas as pd

diaria = pd.read_csv(DADOS + "chuva_diaria_destaque.csv", dtype={"cod": str})
diaria["data"] = pd.to_datetime(diaria["data"])
diaria["ano"] = diaria["data"].dt.year
diaria["mes"] = diaria["data"].dt.month
sh = diaria[diaria["cod"] == "2454012"]          # Santa Helena
print("Dias de chuva diária de Santa Helena:", len(sh))

---

# Capítulo 1. Python com chuva

**Prepara para:** [01_python_basico](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/01_python_basico.ipynb)

### 1.1 Resolvido: a maior sequência de dias secos

Um **veranico** é uma sequência de dias sem chuva no meio da estação chuvosa, e é uma das maiores causas de perda
na soja. O código abaixo percorre os dias de janeiro de 2012 em Santa Helena e conta a maior sequência de dias
secos (chuva menor que 1 mm). A ideia: um contador que cresce a cada dia seco e volta a zero quando chove.

In [ ]:
jan2012 = sh[(sh["ano"] == 2012) & (sh["mes"] == 1)]

sequencia = 0            # dias secos seguidos até agora
maior = 0                # a maior sequência encontrada
for chuva in jan2012["chuva_mm"]:
    if chuva < 1.0:
        sequencia = sequencia + 1      # mais um dia seco
    else:
        sequencia = 0                  # choveu: a sequência recomeça
    if sequencia > maior:
        maior = sequencia              # guarda o recorde

print("Maior sequência de dias secos em jan/2012:", maior, "dias")
print("Chuva total do mês:", round(jan2012["chuva_mm"].sum(), 1), "mm")

### 1.2 Resolvido: transformar o código em função

O mesmo cálculo dentro de uma função, para usar em qualquer mês e em qualquer estação.

In [ ]:
def maior_veranico(chuvas, limiar=1.0):
    """Maior número de dias seguidos com chuva abaixo do limiar (mm)."""
    sequencia = 0
    maior = 0
    for chuva in chuvas:
        if chuva < limiar:
            sequencia = sequencia + 1
        else:
            sequencia = 0
        if sequencia > maior:
            maior = sequencia
    return maior


for ano in [2010, 2011, 2012]:
    dados = sh[(sh["ano"] == ano) & (sh["mes"] == 1)]
    print("Janeiro de", ano, ":", maior_veranico(dados["chuva_mm"]), "dias")

### 1.3 Para pensar e resolver

a) Use `maior_veranico` para achar o maior veranico de **dezembro a fevereiro** da safra 2011/2012 em Santa Helena
(dezembro de 2011 + janeiro e fevereiro de 2012). Dica: filtre as datas com
`sh[(sh["data"] >= "2011-12-01") & (sh["data"] <= "2012-02-29")]`.

b) Mude o limiar para 5 mm. O resultado muda muito? Por que alguém usaria 5 mm em vez de 1 mm?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 2. pandas: somar, agrupar e comparar

**Prepara para:** [02_dados_chuva](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/02_dados_chuva.ipynb)

### 2.1 Resolvido: chuva anual de uma estação, com `groupby`

`groupby("ano")` separa a tabela em pedaços, um por ano; `["chuva_mm"].sum()` soma a chuva de cada pedaço. O
resultado é uma série com um valor por ano.

In [ ]:
anual = sh[sh["ano"] <= 2019].groupby("ano")["chuva_mm"].sum()

print("Média:", round(anual.mean()), "mm")
print("Desvio padrão:", round(anual.std()), "mm")
print("Coeficiente de variação:", round(anual.std() / anual.mean() * 100, 1), "%")
anual.round(0).tail()

### 2.2 Resolvido: anomalia

A **anomalia** é quanto um ano ficou acima ou abaixo da média. Em porcentagem: (ano − média) ÷ média × 100. É a
ideia que está por trás do SPI, que você vai ver no capítulo 5.

In [ ]:
anomalia = (anual - anual.mean()) / anual.mean() * 100

print("Os 3 anos mais secos (anomalia em %):")
print(anomalia.sort_values().head(3).round(0))
print()
print("Os 3 anos mais chuvosos (anomalia em %):")
print(anomalia.sort_values(ascending=False).head(3).round(0))

### 2.3 Resolvido: juntar duas tabelas com `merge`

Muitas vezes a informação está em duas tabelas: a chuva numa, o nome da estação noutra. `merge` junta as duas pela
coluna que elas têm em comum (aqui, `cod`).

In [ ]:
est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str})
media = diaria[diaria["ano"] <= 2019].groupby(["cod", "ano"])["chuva_mm"].sum().groupby("cod").mean().reset_index()

juntas = media.merge(est[["cod", "municipio", "alt_m"]], on="cod")
juntas.round(0)

### 2.4 Para pensar e resolver

a) Calcule a anomalia anual de **Cascavel** e diga se os 3 anos mais secos de Cascavel são os mesmos de Santa
Helena. Por que podem não ser?

b) O coeficiente de variação da chuva **anual** de Santa Helena é bem menor que o CV dos meses de inverno que você
vai ver no notebook 02. Por que a chuva de um ano inteiro varia menos que a de um mês?

In [ ]:
# escreva sua resposta aqui

*Escreva aqui a resposta do item b.*

---

# Capítulo 3. Tendência e ponto de mudança na mão

**Prepara para:** [03_homogeneidade](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/03_homogeneidade.ipynb)

### 3.1 Resolvido: a ideia do Mann-Kendall

O teste compara **todos os pares de anos** (um ano antes, outro depois) e conta: +1 se o depois choveu mais, −1 se
choveu menos. A soma S diz se predominam as subidas (S > 0) ou as descidas (S < 0). Com uma série de 6 valores:

In [ ]:
serie = [1500, 1620, 1580, 1700, 1690, 1750]      # chuva anual inventada (mm)

s = 0
for i in range(len(serie) - 1):
    for j in range(i + 1, len(serie)):            # j sempre depois de i
        if serie[j] > serie[i]:
            s = s + 1
            print(f"ano {i} -> ano {j}: subiu  (+1)")
        elif serie[j] < serie[i]:
            s = s - 1
            print(f"ano {i} -> ano {j}: desceu (-1)")
print("S =", s, "de", len(serie) * (len(serie) - 1) // 2, "pares")

### 3.2 Resolvido: Santa Helena com as funções do projeto

As funções `mann_kendall`, `sen` e `pettitt` (em `src/estatistica.py`) fazem a mesma conta, mais o valor-p.

In [ ]:
import sys
import urllib.request
if Path("../src").exists():
    sys.path.append("../src")
else:
    Path("src").mkdir(exist_ok=True)
    for arquivo in ["fdp.py", "spi.py", "enos.py", "estatistica.py"]:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/src/" + arquivo,
                                   "src/" + arquivo)
    sys.path.append("src")
import estatistica as E

anual_sh = list(anual)                          # a série anual de Santa Helena (capítulo 2)
s, z, p = E.mann_kendall(anual_sh)
print("Mann-Kendall: S =", s, "| p =", round(p, 3))
print("Sen:", round(E.sen(anual_sh) * 10, 1), "mm por década")
pos, k, p_pet = E.pettitt(anual_sh)
print("Pettitt: mudança depois de", list(anual.index)[pos], "| p =", round(p_pet, 3))

### 3.3 Para pensar e resolver

a) Mude a série inventada do item 3.1 para que ela **não** tenha tendência (por exemplo, sobe e desce). Quanto dá S?

b) Por que um teste que só olha "subiu ou desceu" (e não quanto) é útil para chuva, que tem anos extremos?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 4. Uma distribuição ajustada à chuva

**Prepara para:** [04_ajuste_fdp](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/04_ajuste_fdp.ipynb)

### 4.1 Resolvido: histograma e a curva Gama

A chuva de novembro em Santa Helena, um valor por ano. A curva Gama é ajustada aos dados com `stats.gamma.fit`;
`floc=0` fixa o início da curva no zero.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

mensal_sh = sh[sh["ano"] <= 2019].groupby(["ano", "mes"])["chuva_mm"].sum().reset_index()
novembro = mensal_sh.loc[mensal_sh["mes"] == 11, "chuva_mm"]

forma, inicio, escala = stats.gamma.fit(novembro, floc=0)
print(f"alfa (forma) = {forma:.2f}   beta (escala) = {escala:.1f} mm")

x = np.linspace(0, novembro.max() * 1.2, 200)
plt.hist(novembro, bins=10, density=True, color="#e6e5e0", edgecolor="white")
plt.plot(x, stats.gamma.pdf(x, forma, 0, escala), color="#2a78d6", linewidth=2)
plt.xlabel("Chuva de novembro (mm)")
plt.ylabel("Densidade")
plt.show()

### 4.2 Resolvido: probabilidade acumulada

A função de distribuição acumulada (`cdf`) diz a probabilidade de chover **até** um valor. Ela é o primeiro passo do
SPI.

In [ ]:
for chuva in [50, 100, 150, 200, 300]:
    prob = stats.gamma.cdf(chuva, forma, 0, escala)
    print(f"P(chuva de novembro <= {chuva} mm) = {prob:.2f}")

### 4.3 Para pensar e resolver

a) Em quantos anos de cada 10, novembro tem menos de 100 mm em Santa Helena, segundo a Gama?

b) Ajuste a Log-Normal (`stats.lognorm.fit(novembro, floc=0)`) e desenhe as duas curvas juntas. Elas diferem mais no
meio ou nas pontas?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 5. O SPI de um mês, à mão

**Prepara para:** [05_spi](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/05_spi.ipynb)

### 5.1 Resolvido: da chuva ao SPI

Três passos: (1) a chuva do mês; (2) a probabilidade acumulada pela Gama; (3) o valor da normal padrão com a mesma
probabilidade (`stats.norm.ppf`). O resultado é o SPI-1 de novembro.

In [ ]:
for ano in [2005, 2015, 2019]:
    chuva = mensal_sh.loc[(mensal_sh["ano"] == ano) & (mensal_sh["mes"] == 11), "chuva_mm"].iloc[0]
    prob = stats.gamma.cdf(chuva, forma, 0, escala)          # passo 2
    spi_valor = stats.norm.ppf(prob)                         # passo 3
    print(f"novembro de {ano}: {chuva:6.1f} mm -> probabilidade {prob:.2f} -> SPI = {spi_valor:5.2f}")

### 5.2 Resolvido: por que "padronizado"

A normal padrão tem média 0 e desvio padrão 1, em qualquer lugar. Assim, SPI = −1,5 tem o mesmo significado em
Santa Helena e em Curitiba, em janeiro ou em julho: "tão seco quanto acontece em cerca de 7% dos casos".

In [ ]:
for valor in [-2.0, -1.5, -1.0, 0.0, 1.0, 1.5, 2.0]:
    print(f"SPI <= {valor:4.1f}: {stats.norm.cdf(valor) * 100:5.1f}% dos casos")

### 5.3 Para pensar e resolver

a) Calcule o SPI-1 de novembro de **todos** os anos e confira que a média dá perto de 0 e o desvio padrão perto
de 1.

b) Um novembro com 0 mm de chuva quebraria a conta (a Gama não aceita zero). Como o trabalho resolve isso?
(Leia a Equação 5 no TCC2.)

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 6. A regra dos 5 trimestres

**Prepara para:** [06_oni_enos](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/06_oni_enos.ipynb)

### 6.1 Resolvido: média móvel de 3 meses

O ONI é a **média móvel de 3 meses** da anomalia de temperatura do mar. Com uma lista inventada de anomalias
mensais:

In [ ]:
anomalias = [0.2, 0.4, 0.6, 0.8, 0.9, 1.1, 1.0, 0.8, 0.6, 0.3, 0.1, -0.2]

oni = []
for i in range(1, len(anomalias) - 1):
    media = (anomalias[i - 1] + anomalias[i] + anomalias[i + 1]) / 3     # o mês e seus dois vizinhos
    oni.append(round(media, 2))
print("ONI:", oni)

### 6.2 Resolvido: contar trimestres seguidos acima de +0,5

Um El Niño exige **pelo menos 5** valores seguidos ≥ +0,5. O contador cresce enquanto a condição vale e zera quando
ela falha, como no veranico do capítulo 1.

In [ ]:
seguidos = 0
maior = 0
for valor in oni:
    if valor >= 0.5:
        seguidos = seguidos + 1
    else:
        seguidos = 0
    maior = max(maior, seguidos)
print("Maior sequência >= +0,5:", maior, "trimestres ->", "El Niño" if maior >= 5 else "não é El Niño")

### 6.3 Para pensar e resolver

a) Mude a lista de anomalias para que o pico seja igual, mas a sequência acima de +0,5 dure só 4 trimestres. Ainda
é El Niño?

b) Por que a NOAA exige 5 trimestres seguidos, e não apenas um mês acima de +0,5?

In [ ]:
# escreva sua resposta aqui

---

# Capítulo 7. Correlação, autocorrelação e contingência

**Prepara para:** [07_spi_x_enos](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/07_spi_x_enos.ipynb)

### 7.1 Resolvido: a autocorrelação engana

Duas séries **sem nenhuma relação** entre si, mas "suavizadas" (cada valor é a soma dos 12 anteriores, como o
SPI-12). A correlação entre elas sai alta com frequência, só por acaso. É por isso que o trabalho usa o bootstrap
em blocos em vez do teste t comum.

In [ ]:
gerador = np.random.default_rng(3)
correlacoes = []
for tentativa in range(200):
    a = np.convolve(gerador.normal(size=480), np.ones(12), mode="valid")   # soma móvel de 12
    b = np.convolve(gerador.normal(size=480), np.ones(12), mode="valid")
    correlacoes.append(np.corrcoef(a, b)[0, 1])
correlacoes = np.array(correlacoes)
print("Em 200 pares de séries independentes:")
print("  correlação maior que 0,2 em módulo:", np.mean(np.abs(correlacoes) > 0.2) * 100, "% das vezes")
print("  sem suavizar, isso aconteceria em menos de 0,1% das vezes (com 480 valores)")

### 7.2 Resolvido: tabela de contingência com números simples

Suponha 40 primaveras: 10 de La Niña, 10 de El Niño, 20 neutras. Contamos quantas foram secas em cada fase.

In [ ]:
import pandas as pd

tabela = pd.DataFrame({"seco": [5, 1, 3], "normal": [4, 6, 15], "úmido": [1, 3, 2]},
                      index=["La Niña", "El Niño", "Neutro"])
print(tabela)

p_seco_lanina = tabela.loc["La Niña", "seco"] / tabela.loc["La Niña"].sum() * 100
p_seco_clima = tabela["seco"].sum() / tabela.values.sum() * 100
print()
print(f"P(seco | La Niña) = {p_seco_lanina:.0f}%")
print(f"P(seco)           = {p_seco_clima:.0f}%")
print(f"Risco relativo    = {p_seco_lanina / p_seco_clima:.1f}")

### 7.3 Para pensar e resolver

a) Com a tabela do item 7.2, calcule P(úmido | El Niño), P(úmido) e o risco relativo. A resposta é simétrica à da
La Niña?

b) Explique, em duas frases para um produtor de soja, o que quer dizer "risco relativo de 2,5".

In [ ]:
# escreva sua resposta aqui

*Escreva aqui a resposta do item b.*